In [204]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window


spark = (
    SparkSession.builder
        .appName("etl-datalake")
        .master("local[*]")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [ ]:
# Carregar os dados dos arquivos CSV para DataFrames do Spark - Camada de Ingestão - Raw Zone
df_rh_folha = spark.read \
    .option("header", True) \
    .csv("data/raw/rh_folha_pessoas")\
    .withColumn("origem_registro", F.lit("rh_folha"))

df_esocial = spark.read \
    .option("header", True) \
    .csv("data/raw/esocial_pessoas")\
    .withColumn("origem_registro", F.lit("esocial")) \

df_gestao = spark.read \
    .option("header", True) \
    .csv("data/raw/gestao_pessoas")\
    .withColumn("origem_registro", F.  lit("gestao")) 

In [206]:
# Visualizar os schemas dos DataFrames
df_rh_folha.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4500   |2023-01-01   |rh_folha       |
|55555555505|Ricardo Alves|Rua E, 500|2021-06-18   |5200   |2023-01-01   |rh_folha       |
|33333333303|Carlos Melo  |Rua C, 300|2018-11-20   |6000   |2023-01-01   |rh_folha       |
|22222222202|Maria Souza  |Rua B, 200|2019-03-15   |7000   |2023-01-01   |rh_folha       |
|11111111101|João Silva   |Rua A, 100|2020-01-10   |5000   |2023-01-01   |rh_folha       |
+-----------+-------------+----------+-------------+-------+-------------+---------------+



In [207]:
# Avaliar a quantidade de valores nulos em cada coluna do DataFrame
df_rh_folha.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in df_rh_folha.columns
]).show()

+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|cpf_nulls|nome_nulls|endereco_nulls|data_admissao_nulls|salario_nulls|data_registro_nulls|origem_registro_nulls|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|        0|         0|             0|                  0|            0|                  0|                    0|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+



In [208]:
# Avaliar a quantidade de valores duplicados com base na combinação de nome e data de admissão
df_rh_folha.groupBy("nome", "data_admissao") \
  .count() \
  .filter("count > 1") \
  .show()

+----+-------------+-----+
|nome|data_admissao|count|
+----+-------------+-----+
+----+-------------+-----+



In [209]:
# Retirando os registros duplicados, mantendo apenas o mais recente com base na data de registro
window_spec = Window.partitionBy("nome").orderBy(F.col("data_registro").desc())
df_ranked = df_rh_folha.withColumn(
    "rn",
    F.row_number().over(window_spec)
)
df_rh_folha = df_ranked.filter("rn = 1").drop("rn")
df_rh_folha.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|33333333303|Carlos Melo  |Rua C, 300|2018-11-20   |6000   |2023-01-01   |rh_folha       |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4500   |2023-01-01   |rh_folha       |
|11111111101|João Silva   |Rua A, 100|2020-01-10   |5000   |2023-01-01   |rh_folha       |
|22222222202|Maria Souza  |Rua B, 200|2019-03-15   |7000   |2023-01-01   |rh_folha       |
|55555555505|Ricardo Alves|Rua E, 500|2021-06-18   |5200   |2023-01-01   |rh_folha       |
+-----------+-------------+----------+-------------+-------+-------------+---------------+



In [210]:
#########################################################################################################################

In [211]:
df_esocial.show(5, False)

+-----------+--------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome          |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+--------------+----------+-------------+-------+-------------+---------------+
|77777777707|Bruno Costa   |Rua G, 700|2020-09-10   |3900   |2023-03-01   |esocial        |
|88888888808|Amanda Letícia|Rua H, 800|2019-12-04   |5300   |2023-03-01   |esocial        |
|66666666606|Ana Lima      |Rua F, 600|2021-07-01   |4000   |2023-03-01   |esocial        |
|33333333303|Carlos Melo   |Rua C, 305|2018-11-20   |6100   |2023-03-01   |esocial        |
|44444444404|Fernanda Lima |Rua D, 400|2022-02-05   |4500   |2023-03-01   |esocial        |
+-----------+--------------+----------+-------------+-------+-------------+---------------+
only showing top 5 rows



In [212]:
# Avaliar a quantidade de valores nulos em cada coluna do DataFrame
df_esocial.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in df_esocial.columns
]).show()

+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|cpf_nulls|nome_nulls|endereco_nulls|data_admissao_nulls|salario_nulls|data_registro_nulls|origem_registro_nulls|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|        0|         0|             0|                  0|            0|                  0|                    0|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+



In [213]:
# Avaliar a quantidade de valores duplicados com base na combinação de nome e data de admissão
df_esocial.groupBy("nome", "data_admissao") \
  .count() \
  .filter("count > 1") \
  .show()

+----+-------------+-----+
|nome|data_admissao|count|
+----+-------------+-----+
+----+-------------+-----+



In [214]:
# Retirando os registros duplicados, mantendo apenas o mais recente com base na data de registro
window_spec = Window.partitionBy("cpf").orderBy(F.col("data_registro").desc())
df_ranked = df_esocial.withColumn(
    "rn",
    F.row_number().over(window_spec)
)
df_esocial = df_ranked.filter("rn = 1").drop("rn")
df_esocial.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|11111111101|João Silva   |Rua A, 100|2020-01-10   |4800   |2023-03-01   |esocial        |
|33333333303|Carlos Melo  |Rua C, 305|2018-11-20   |6100   |2023-03-01   |esocial        |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4500   |2023-03-01   |esocial        |
|66666666606|Ana Lima     |Rua F, 600|2021-07-01   |4000   |2023-03-01   |esocial        |
|77777777707|Bruno Costa  |Rua G, 700|2020-09-10   |3900   |2023-03-01   |esocial        |
+-----------+-------------+----------+-------------+-------+-------------+---------------+
only showing top 5 rows



In [215]:
df_gestao.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|88888888808|Juliana Rocha|Rua H, 800|2019-12-01   |5300   |2023-05-01   |gestao         |
|55555555505|Ricardo Alves|Rua E, 550|2021-06-18   |5000   |2023-05-01   |gestao         |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4700   |2023-05-01   |gestao         |
|33333333303|Carlos Melo  |Rua C, 305|2018-11-20   |6000   |2023-05-01   |gestao         |
|11111111101|João Silva   |Rua A, 120|2020-01-10   |5100   |2023-05-01   |gestao         |
+-----------+-------------+----------+-------------+-------+-------------+---------------+



In [216]:
# Avaliar a quantidade de valores nulos em cada coluna do DataFrame
df_gestao.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in df_gestao.columns
]).show()

+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|cpf_nulls|nome_nulls|endereco_nulls|data_admissao_nulls|salario_nulls|data_registro_nulls|origem_registro_nulls|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+
|        0|         0|             0|                  0|            0|                  0|                    0|
+---------+----------+--------------+-------------------+-------------+-------------------+---------------------+



In [217]:
# Avaliar a quantidade de valores duplicados com base na combinação de nome e data de admissão
df_gestao.groupBy("cpf", "data_admissao") \
  .count() \
  .filter("count > 1") \
  .show()

+---+-------------+-----+
|cpf|data_admissao|count|
+---+-------------+-----+
+---+-------------+-----+



In [218]:
# Retirando os registros duplicados, mantendo apenas o mais recente com base na data de registro
window_spec = Window.partitionBy("cpf").orderBy(F.col("data_registro").desc())
df_ranked = df_gestao.withColumn(
    "rn",
    F.row_number().over(window_spec)
)
df_gestao = df_ranked.filter("rn = 1").drop("rn")
df_gestao.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|11111111101|João Silva   |Rua A, 120|2020-01-10   |5100   |2023-05-01   |gestao         |
|33333333303|Carlos Melo  |Rua C, 305|2018-11-20   |6000   |2023-05-01   |gestao         |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4700   |2023-05-01   |gestao         |
|55555555505|Ricardo Alves|Rua E, 550|2021-06-18   |5000   |2023-05-01   |gestao         |
|88888888808|Juliana Rocha|Rua H, 800|2019-12-01   |5300   |2023-05-01   |gestao         |
+-----------+-------------+----------+-------------+-------+-------------+---------------+



In [219]:
df_rh_folha.printSchema()

root
 |-- cpf: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- endereco: string (nullable = true)
 |-- data_admissao: string (nullable = true)
 |-- salario: string (nullable = true)
 |-- data_registro: string (nullable = true)
 |-- origem_registro: string (nullable = false)



In [220]:
df_gestao.printSchema()


root
 |-- cpf: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- endereco: string (nullable = true)
 |-- data_admissao: string (nullable = true)
 |-- salario: string (nullable = true)
 |-- data_registro: string (nullable = true)
 |-- origem_registro: string (nullable = false)



In [221]:
df_esocial.printSchema()


root
 |-- cpf: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- endereco: string (nullable = true)
 |-- data_admissao: string (nullable = true)
 |-- salario: string (nullable = true)
 |-- data_registro: string (nullable = true)
 |-- origem_registro: string (nullable = false)



In [222]:
df = df_rh_folha.unionByName(df_esocial).unionByName(df_gestao)
df.show(5, False)

+-----------+-------------+----------+-------------+-------+-------------+---------------+
|cpf        |nome         |endereco  |data_admissao|salario|data_registro|origem_registro|
+-----------+-------------+----------+-------------+-------+-------------+---------------+
|33333333303|Carlos Melo  |Rua C, 300|2018-11-20   |6000   |2023-01-01   |rh_folha       |
|44444444404|Fernanda Lima|Rua D, 400|2022-02-05   |4500   |2023-01-01   |rh_folha       |
|11111111101|João Silva   |Rua A, 100|2020-01-10   |5000   |2023-01-01   |rh_folha       |
|22222222202|Maria Souza  |Rua B, 200|2019-03-15   |7000   |2023-01-01   |rh_folha       |
|55555555505|Ricardo Alves|Rua E, 500|2021-06-18   |5200   |2023-01-01   |rh_folha       |
+-----------+-------------+----------+-------------+-------+-------------+---------------+
only showing top 5 rows



In [223]:
# 1. Criar prioridade de fonte
df = df.withColumn(
    "prioridade",
    F.when(F.col("origem_registro") == "esocial", 1)
    .when(F.col("origem_registro") == "rh_folha", 2)
    .otherwise(3)
)

# 2. Definir janela global
window_spec = Window.partitionBy("cpf").orderBy(
    F.col("prioridade"),
    F.col("data_registro").desc()
)

# 3. Rankear e definir registro atual
df_final = df.withColumn(
    "rn",
    F.row_number().over(window_spec)
).withColumn(
    "registro_atual",
    F.when(F.col("rn") == 1, "S").otherwise("N")
).drop("rn", "prioridade")

In [224]:
df_final.groupBy("nome", "data_admissao") \
  .count() \
  .filter("count > 1") \
  .show()

+-------------+-------------+-----+
|         nome|data_admissao|count|
+-------------+-------------+-----+
|Ricardo Alves|   2021-06-18|    2|
|   João Silva|   2020-01-10|    3|
|  Carlos Melo|   2018-11-20|    3|
|Fernanda Lima|   2022-02-05|    3|
+-------------+-------------+-----+



In [225]:
df_final.show(20,False)

+-----------+--------------+----------+-------------+-------+-------------+---------------+--------------+
|cpf        |nome          |endereco  |data_admissao|salario|data_registro|origem_registro|registro_atual|
+-----------+--------------+----------+-------------+-------+-------------+---------------+--------------+
|11111111101|João Silva    |Rua A, 100|2020-01-10   |4800   |2023-03-01   |esocial        |S             |
|11111111101|João Silva    |Rua A, 100|2020-01-10   |5000   |2023-01-01   |rh_folha       |N             |
|11111111101|João Silva    |Rua A, 120|2020-01-10   |5100   |2023-05-01   |gestao         |N             |
|22222222202|Maria Souza   |Rua B, 200|2019-03-15   |7000   |2023-01-01   |rh_folha       |S             |
|33333333303|Carlos Melo   |Rua C, 305|2018-11-20   |6100   |2023-03-01   |esocial        |S             |
|33333333303|Carlos Melo   |Rua C, 300|2018-11-20   |6000   |2023-01-01   |rh_folha       |N             |
|33333333303|Carlos Melo   |Rua C, 30

In [226]:
# Verificação final de registros e colunas
num_linhas = df_final.count()
num_colunas = len(df_final.columns)

print(f"Linhas: {num_linhas}")
print(f"Colunas: {num_colunas}")

Linhas: 16
Colunas: 8


In [227]:
# Avaliar a quantidade de valores duplicados com base na combinação de nome e data de admissão
df_final.groupBy("nome", "data_admissao") \
  .count() \
  .filter("count > 1") \
  .show()

+-------------+-------------+-----+
|         nome|data_admissao|count|
+-------------+-------------+-----+
|Ricardo Alves|   2021-06-18|    2|
|   João Silva|   2020-01-10|    3|
|  Carlos Melo|   2018-11-20|    3|
|Fernanda Lima|   2022-02-05|    3|
+-------------+-------------+-----+



In [228]:
# Verificação final de registros e colunas
num_linhas = df.count()
num_colunas = len(df.columns)

print(f"Linhas: {num_linhas}")
print(f"Colunas: {num_colunas}")

Linhas: 16
Colunas: 8
